# **Async Python - A Practical Guide for Agentic AI**

## 1. Introduction

### 1.1. Why Async Matters (Especially for Agentic Systems)

Agentic AI systems are inherently **I/O-bound and concurrent**:

* Calling multiple LLM APIs
* Querying vector databases
* Fetching documents (RAG)
* Tool execution (web, DBs, APIs)
* Multi-step reasoning chains

If you run these sequentially, latency compounds linearly.

**Async solves this by:**

* Letting one task run while another waits (non-blocking execution)
* Maximizing throughput without heavy threads/process overhead
* Enabling fine-grained orchestration of agent workflows

### 1.2. Core Mental Model

#### Synchronous Execution (Blocking)

```text
Task A → wait → Task B → wait → Task C
```

#### Asynchronous Execution (Non-blocking)

```text
Task A ┐
Task B ├→ interleaved execution (event loop)
Task C ┘
```

Async Python uses:

* **Event loop** → scheduler
* **Coroutines** → async functions
* **Awaitables** → things you can pause on



## 2. Async vs Threads vs Multiprocessing

When you need concurrency in Python, you’re choosing between three fundamentally different execution models: **asynchronous I/O (async)**, **threading**, and **multiprocessing**. The right choice depends on the nature of your workload—specifically whether it’s I/O-bound or CPU-bound—and how you want to manage parallelism.

---

#### 1. Async (Asynchronous I/O)

**Mental model:** *One thread, many tasks, cooperative scheduling.*

Async in Python (via `asyncio`) is designed for **I/O-bound workloads**—situations where your program spends most of its time waiting (e.g., network calls, disk I/O, APIs).

* Uses a single thread and a single process.
* Tasks explicitly yield control (`await`) when they are waiting.
* The event loop schedules and resumes tasks efficiently.
* No true parallelism—only concurrency.

**Key implications:**

* Extremely lightweight compared to threads/processes.
* No context-switching overhead at the OS level.
* Avoids many synchronization issues (no shared-state race conditions by default).
* Requires libraries that support async (not all do).

**Best for:** high-throughput I/O systems (APIs, web scraping, streaming, async pipelines).

---

#### 2. Threads (Multithreading)

**Mental model:** *Multiple threads within one process, preemptively scheduled.*

Threads allow multiple execution paths within the same process. However, in Python, the **Global Interpreter Lock (GIL)** is a critical constraint.

* Multiple threads share the same memory space.
* The OS scheduler switches between threads.
* The GIL ensures only one thread executes Python bytecode at a time.

**Key implications:**

* Useful for I/O-bound tasks (threads can run while others are blocked).
* Not effective for CPU-bound parallelism due to the GIL.
* Shared memory makes communication easy—but introduces risks (race conditions, deadlocks).
* Debugging can become complex.

**Best for:** I/O-bound tasks when async is not viable (e.g., blocking libraries, legacy code).

---

#### 3. Multiprocessing

**Mental model:** *Multiple independent processes, true parallelism.*

Multiprocessing bypasses the GIL by using separate processes, each with its own Python interpreter and memory space.

* True parallel execution across CPU cores.
* No shared memory by default (communication via IPC: pipes, queues, shared memory).
* Higher overhead (process creation, data serialization).

**Key implications:**

* Ideal for CPU-bound tasks (e.g., numerical computation, data processing).
* More resource-intensive than threads or async.
* Requires careful design for inter-process communication.

**Best for:** CPU-heavy workloads where parallel execution provides real speedup.

---

### Summary Comparison

| Aspect      | Async                   | Threads                | Multiprocessing           |
| ----------- | ----------------------- | ---------------------- | ------------------------- |
| Parallelism | No (concurrency only)   | Limited (GIL-bound)    | Yes (true parallelism)    |
| Best for    | I/O-bound               | I/O-bound (blocking)   | CPU-bound                 |
| Memory      | Shared (single thread)  | Shared                 | Separate                  |
| Overhead    | Low                     | Medium                 | High                      |
| Complexity  | Medium (async patterns) | High (synchronization) | High (IPC, serialization) |
| GIL impact  | Avoided (single thread) | Constrains execution   | Avoided (separate procs)  |

---

### Practical Heuristic

* If your bottleneck is **waiting on external resources** → use **async**.
* If you must use **blocking I/O libraries** → use **threads**.
* If your bottleneck is **CPU computation** → use **multiprocessing**.

---

### Final Insight

These models are not mutually exclusive. In production systems (e.g., data pipelines or agentic architectures), it’s common to combine them:

* Async for orchestration and I/O
* Threads for legacy/blocking integrations
* Multiprocessing for heavy compute stages

Understanding their trade-offs lets you design systems that are both efficient and predictable.


## 3. The Foundation

### 3.1. Coroutines

A coroutine is just a function defined with `async def`.

```python
async def fetch_data():
    return "data"
```

Key points:

* Calling it does **not execute it**
* It returns a **coroutine object**

To run it:

```python
import asyncio

async def main():
    result = await fetch_data()
    print(result)

asyncio.run(main())
```

---

### 3.2. `await`: The Yield Point

`await` pauses execution and yields control back to the event loop.

```python
async def task():
    print("Start")
    await asyncio.sleep(2)
    print("End")
```

While waiting:

* The event loop runs other tasks
* No thread is blocked

---

### 3.3. Event Loop (Execution Engine)

The event loop:

* Tracks pending tasks
* Switches execution when tasks `await`
* Runs until all tasks complete

You usually interact with it via:

```python
asyncio.run(main())
```

---

### 3.4 Running Multiple Tasks Concurrently

#### `asyncio.gather` (Most Common)

```python
async def task(n):
    await asyncio.sleep(1)
    return f"Task {n}"

async def main():
    results = await asyncio.gather(
        task(1),
        task(2),
        task(3)
    )
    print(results)
```

Execution:

* All tasks start immediately
* Run concurrently
* Results returned in order

---

#### `asyncio.create_task` (More Control)

```python
async def main():
    t1 = asyncio.create_task(task(1))
    t2 = asyncio.create_task(task(2))

    result1 = await t1
    result2 = await t2
```

Use when:

* You want to start tasks early
* You control when to await

We're basicaly scheduling a coroutine to run as quick as possible.

As soon as a coroutine is sleeping or idle waiting something that's not in control of our program the event loop will move on and start another task.

---

### 3.5. Concurrency vs Parallelism

Async = **concurrency**, not parallelism.

* Single thread
* Cooperative multitasking
* Best for **I/O-bound workloads**

For CPU-bound:

* Use multiprocessing

---

### 3.6. Real-World Pattern: API Calls (Agent Use Case)

```python
import asyncio
import random

async def call_llm(prompt):
    await asyncio.sleep(random.uniform(0.5, 1.5))
    return f"Response to {prompt}"

async def main():
    prompts = ["A", "B", "C"]

    responses = await asyncio.gather(
        *[call_llm(p) for p in prompts]
    )

    print(responses)

asyncio.run(main())
```

This mirrors:

* Multi-agent querying
* Tool parallelization
* Batch reasoning

---

### 3.7. Async + Rate Limiting (Critical for LLM Systems)

Without control, async can overwhelm APIs.

#### Semaphore Pattern

```python
semaphore = asyncio.Semaphore(2)

async def safe_call(prompt):
    async with semaphore:
        return await call_llm(prompt)
```

This ensures:

* Max 2 concurrent requests

---

### 3.8. Async Iteration

Useful for streaming or pipelines.

```python
async def async_generator():
    for i in range(3):
        await asyncio.sleep(1)
        yield i

async def main():
    async for item in async_generator():
        print(item)
```

Use cases:

* Streaming LLM tokens
* Incremental RAG retrieval
* Tool pipelines

---

### 3.9. Error Handling in Async

#### With `gather`

```python
results = await asyncio.gather(
    task(1),
    task(2),
    return_exceptions=True
)
```

Without `return_exceptions=True`:

* One failure cancels all

---

### 3.10. Cancellation

Agents often need timeouts or early stopping.

```python
task = asyncio.create_task(call_llm("test"))

await asyncio.sleep(1)
task.cancel()
```

---

### 3.11. Timeouts

```python
try:
    result = await asyncio.wait_for(call_llm("test"), timeout=2)
except asyncio.TimeoutError:
    print("Timed out")
```

Critical for:

* External APIs
* Tool execution

---

### 3.12. Blocking Code Pitfall

Async breaks if you call blocking functions.

Bad:

```python
import time

async def task():
    time.sleep(2)  # blocks everything
```

Fix:

```python
await asyncio.to_thread(time.sleep, 2)
```

---

### 3.13. Async + Threads (Hybrid Model)

Use when:

* Library is not async
* CPU work is small

```python
await asyncio.to_thread(sync_function)
```

---

### 3.14. Structured Concurrency (Python 3.11+)

#### Task Groups

```python
async def main():
    async with asyncio.TaskGroup() as tg:
        tg.create_task(task(1))
        tg.create_task(task(2))
```

Benefits:

* Safer lifecycle management
* Automatic cancellation on failure

---

### 3.15. Async in Agentic AI Architectures

#### Pattern 1: Parallel Tool Execution

```python
await asyncio.gather(
    search_tool(query),
    db_tool(query),
    llm_tool(query)
)
```

---

#### Pattern 2: Multi-Agent Systems

```python
agents = [agent1(), agent2(), agent3()]

results = await asyncio.gather(*agents)
```

---

#### Pattern 3: RAG Pipelines

```python
docs, embeddings = await asyncio.gather(
    retrieve_docs(query),
    embed_query(query)
)
```

---

#### Pattern 4: Orchestrated Reasoning

```python
task1 = asyncio.create_task(step1())
task2 = asyncio.create_task(step2())

await task1
await task2
```

---

### 3.16. Performance Intuition

| Workload Type            | Best Approach   |
| ------------------------ | --------------- |
| I/O-bound (APIs, DBs)    | Async           |
| CPU-bound (ML inference) | Multiprocessing |
| Mixed                    | Async + threads |

---

### 3.17. Common Anti-Patterns

#### ❌ Awaiting sequentially

```python
await task1()
await task2()
```

#### ✅ Use gather

```python
await asyncio.gather(task1(), task2())
```

---

#### ❌ Over-parallelization

* Causes rate limits
* Resource exhaustion

---

#### ❌ Mixing sync + async blindly

* Leads to blocking

---

### 3.18. Minimal Mental Checklist

When designing an async agent system:

1. Is this I/O-bound? → use async
2. Can tasks run independently? → use `gather`
3. Is there a rate limit? → use semaphore
4. Can it fail? → handle exceptions
5. Can it hang? → add timeout

---

### 3.19. Key Takeaways

* Async Python is about **efficient waiting**, not speed by computation
* The event loop orchestrates all concurrency
* `await` is the key yield mechanism
* `gather` is your default concurrency primitive
* Critical for **scalable Agentic AI systems**

---


## 4. Caveats on jupyter notebooks

Async behaves differently in Jupyter notebooks because the notebook **already runs an event loop**. That creates a few practical constraints:

---

### 1. You cannot use `asyncio.run()`

In scripts:

```python
asyncio.run(main())
```

In Jupyter:

* This raises: `RuntimeError: asyncio.run() cannot be called from a running event loop`

**Why:** Jupyter (via IPython) already has an active loop.

**What to do instead:**

```python
await main()
```

---

### 2. Top-level `await` is allowed (and preferred)

Jupyter supports:

```python
await some_async_function()
```

This is not valid in standard Python scripts, but it is the **correct pattern in notebooks**.

---

### 3. Event loop is shared across cells

* Tasks created in one cell may still be running in another
* State is not isolated per cell

**Implication:**

* Harder to reason about lifecycle
* Bugs can look “non-deterministic”

---

### 4. Background tasks can leak

```python
asyncio.create_task(some_task())
```

If you don’t await or track it:

* It keeps running in the background
* You can get unexpected outputs later

---

### 5. Re-running cells can duplicate tasks

If you define and run:

```python
asyncio.create_task(worker())
```

Then re-run the cell:

* You now have multiple workers running concurrently

---

### 6. Blocking calls break everything (same as normal async, but worse)

If you do:

```python
time.sleep(2)
```

* You block the notebook UI
* Kernel appears frozen

Use:

```python
await asyncio.sleep(2)
```

---

### 7. Debugging is harder

* Stack traces are less clear
* Errors inside tasks may not surface immediately
* Silent failures are common if tasks aren’t awaited

---

### 8. Some libraries behave differently

Libraries expecting to control the event loop (e.g., older async frameworks) may:

* Fail
* Require patches like `nest_asyncio` (not recommended for production thinking)

---

### Minimal Best Practices for Notebooks

* Use `await` directly (never `asyncio.run`)
* Prefer `asyncio.gather` over unmanaged `create_task`
* Avoid background tasks unless necessary
* Be careful when re-running cells
* Keep async examples **self-contained per cell**

---

### Bottom Line

Jupyter is great for **learning and prototyping async**, but:

* It hides event loop complexity
* It can mask lifecycle issues

For anything production-like (e.g., Agentic systems), always validate behavior in a **real Python runtime**.


## 5. Where await is used? With what type of function?

`await` is very constrained in Python—it only works in a specific context and with specific types of objects.

---

### 1. Where `await` can be used

`await` can **only appear inside an `async def` function**.

So this is valid:

```python
async def main():
    result = await Runner.run(...)
```

This is invalid:

```python
def main():
    result = await Runner.run(...)  # ❌ SyntaxError
```

Reason: `await` requires an **active event loop context**, which only exists inside a coroutine.

---

### 2. What you can `await`

You can only `await` **awaitable objects**. There are three main categories:

#### (1) Coroutines (most common)

Functions defined with `async def`:

```python
async def foo():
    return 42

result = await foo()
```

In your code:

```python
result = await Runner.run(...)
```

→ `Runner.run()` returns a **coroutine**, so it is awaitable.

---

#### (2) Tasks

A `Task` wraps a coroutine and schedules it:

```python
task = asyncio.create_task(foo())
result = await task
```

Used when you want concurrency (run now, await later).

---

#### (3) Futures

Lower-level primitive (used internally by libraries):

```python
future = some_async_lib_call()
result = await future
```

Most of the time, you won’t create these manually.

---

### 3. What happens if you `await`

When Python hits:

```python
result = await Runner.run(...)
```

It:

1. Calls `Runner.run(...)` → gets a coroutine
2. Suspends execution of the current function
3. Hands control back to the event loop
4. Resumes when the result is ready

This is **cooperative multitasking**, not parallel execution.

---

### 4. What you cannot `await`

You cannot `await`:

* Regular functions
* Plain values
* Blocking I/O functions

Example:

```python
def foo():
    return 42

await foo()  # ❌ TypeError
```

---

### 5. Practical rule (use this mentally)

If a function is:

* defined with `async def` → **must use `await`**
* defined with `def` → **cannot use `await`**

---

### 6. Applied to your code

```python
result = await Runner.run(agent, "When did the Roman Empire fall?")
```

This tells you:

* `Runner.run` is async
* It likely performs network I/O
* It returns a coroutine → resolved via `await`

---

### Bottom line

`await` is:

* **Syntactically restricted** → only inside `async def`
* **Semantically restricted** → only works on *awaitables* (coroutines, tasks, futures)
* **Operationally important** → it prevents blocking and enables concurrency

---

If you want to go one level deeper, the next useful concept is:
**difference between “calling” a coroutine vs “awaiting” it**—that’s where many subtle bugs come from in agent pipelines.
